# Week 6 — Core Features: Full Agent Loop
## Premier League Research Agent
**Goal:** Build the complete agent with multi-tool reasoning, session memory, hook logging, and Pydantic structured outputs.

In [ ]:
from IPython.display import display, Markdown
import os
import sys
import json
import importlib
import pandas as pd
sys.path.insert(0, '.')
from dotenv import load_dotenv
import src.agent as agent_module
importlib.reload(agent_module)
from src.memory import SessionMemory
from src.hooks import get_logs
from src.tools.live_tool import get_standings, get_top_scorers
from src.tools.rag_tool import search_historical_stats
from src.schemas.player_stats import PlayerMatchStats
from src.schemas.standings import CurrentStandings, TeamStanding
from src.agent import run_agent, run_fallback_agent
from pydantic import ValidationError
from openai import OpenAI
from src.constants import (
    AGENT_COLOR, USER_COLOR, RESET, BOLD,
    INFO_COLOR, SUCCESS_COLOR, ERROR_COLOR, CYAN
)

run_agent = agent_module.run_agent

load_dotenv()

import os
if os.path.exists('logs/agent_trace.log'):
    os.remove('logs/agent_trace.log')
    print("Cleared previous logs.")

memory = SessionMemory()
print(f"{CYAN}{BOLD}Week 6 - Core Agent Features{RESET}")
print("Fresh session memory initialized.")
print("Logs cleared.")
print("All imports successful.")

In [ ]:
print(f"{CYAN}{BOLD}Agent Demo: Live Questions{RESET}\n")

live_questions = [
    "Who is top of the Premier League right now?",
    "Who is the leading scorer this season?",
    "What were the most recent Premier League results?",
]

for question in live_questions:
    print(f"{USER_COLOR}{BOLD}You:{RESET} {question}")
    print(f"{INFO_COLOR}{'─' * 60}{RESET}")
    try:
        answer = run_agent(question, memory)
        if answer and len(answer) > 20:
            print(f"{AGENT_COLOR}{BOLD}Agent:{RESET}")
            display(Markdown(answer))
            print()
        else:
            print(f"{ERROR_COLOR}Agent returned short response.{RESET}\n")
    except Exception as e:
        print(f"{ERROR_COLOR}Error: {e}{RESET}\n")
    print()

In [ ]:
from IPython.display import display, Markdown

print(f"{CYAN}{BOLD}Agent Demo: Historical Questions{RESET}\n")

historical_questions = [
    "Show Liverpool vs Manchester City recent Premier League meetings",
    "Show Arsenal vs Chelsea historical results",
    "Show Manchester United vs Liverpool head to head"
]

for question in historical_questions:
    print(f"{USER_COLOR}{BOLD}You:{RESET} {question}")
    print(f"{INFO_COLOR}{'─' * 60}{RESET}")
    try:
        answer = run_agent(question, memory)
        if answer and len(answer) > 20:
            display(Markdown(answer))
        else:
            print(f"{ERROR_COLOR}Agent returned short response.{RESET}\n")
    except Exception as e:
        print(f"{ERROR_COLOR}Error: {e}{RESET}\n")
    print()

In [ ]:
from IPython.display import display, Markdown
import time

print(f"{CYAN}{BOLD}Agent Demo: News Questions{RESET}\n")

news_questions = [
    "What is the latest Premier League transfer news 2026?",
    "Premier League injury news August 2026",
]

for question in news_questions:
    print(f"{USER_COLOR}{BOLD}You:{RESET} {question}")
    print(f"{INFO_COLOR}{'─' * 60}{RESET}")
    try:
        answer = run_agent(question, memory)
        if answer and len(answer) > 20:
            print(f"{AGENT_COLOR}{BOLD}Agent:{RESET}")
            display(Markdown(answer))
        else:
            print(f"{INFO_COLOR}Using fallback...{RESET}")
            answer = run_fallback_agent(question, memory)
            print(f"{AGENT_COLOR}{BOLD}Agent:{RESET}")
            display(Markdown(answer))
    except Exception as e:
        print(f"{ERROR_COLOR}Error: {e}{RESET}")
    time.sleep(1)
    print()

In [ ]:
print(f"{CYAN}{BOLD}Web Search Accuracy Analysis{RESET}\n")

accuracy_note = """
## Known Limitation: Web Search Accuracy

Web search results for transfer news have known accuracy issues:

### What We Found
- Youri Tielemans shown as joining Aston Villa — he actually LEFT
- Transfer fees and destinations conflict between sources
- Injury reports mix pre-season and in-season data

### Why This Happens
| Reason | Explanation |
|--------|-------------|
| Mixed timeframes | Results combine 2025 and 2026 window news |
| Unverified sources | Rumours shown alongside confirmed deals |
| No fact checking | Brave Search has no quality filter |
| Stale cache | Some results are from old cached pages |

### Mitigation Applied
- Year automatically appended to all queries
- Results labeled as unverified in agent responses  
- Agent told to say "unconfirmed" for transfer news

### What Would Fix This Properly
- Use official Premier League transfer API
- Add a fact-checking step against football-data.org
- Cross-reference multiple sources before returning

*This is a documented limitation — web search for sports news
requires additional verification layers in production systems.*
"""
display(Markdown(accuracy_note))

In [ ]:
print(f"{CYAN}{BOLD}Data Quality Analysis - Web Search Accuracy{RESET}\n")
print(f"{USER_COLOR}Observation:{RESET}")
print("""
The Brave Search tool returns real web results but they have limitations:

1. MIXED TIMEFRAMES - Results combine transfers from different summers
   (2025 and 2026) without clear distinction

2. CONFLICTING SOURCES - Different news sites report different destinations
   for the same player

3. UNVERIFIED DATA - Web scraping picks up rumours alongside confirmed deals

4. NO GROUND TRUTH - Unlike StatsBomb (verified match data) or
   football-data.org (official API), Brave Search has no quality filter

This is a real-world RAG/agent limitation - web search tools
require additional validation layers in production systems.

Mitigation strategies:
- Add date filtering to search queries
- Cross-reference multiple sources
- Flag search results as unverified
- Use official APIs where possible (football-data.org for transfers)
""")
print(f"{INFO_COLOR}This is documented as a known limitation in the reflection.{RESET}")

In [ ]:
print(f"{CYAN}{BOLD}Agent Demo: Multi-Hop Question{RESET}\n")
print(f"{INFO_COLOR}This requires multiple tools in sequence{RESET}\n")

question = "Compare Jamie Vardy 2015-16 historical stats with who is currently top scorer this season"

print(f"{USER_COLOR}{BOLD}You:{RESET} {question}")
print(f"{INFO_COLOR}{'─' * 60}{RESET}")
try:
    answer = run_agent(question, memory)
    if answer and len(answer) > 20:
        print(f"{AGENT_COLOR}{BOLD}Agent:{RESET}")
        display(Markdown(answer))
        print()
    else:
        print(f"{ERROR_COLOR}Agent returned short response.{RESET}\n")
except Exception as e:
    print(f"{ERROR_COLOR}Error: {e}{RESET}\n")

In [ ]:
print(f"{CYAN}{BOLD}Session Memory After Conversation{RESET}\n")
facts = memory.get_facts()
if facts == 'No facts stored yet.':
    print(f"{INFO_COLOR}No facts stored yet. Run agent cells first.{RESET}")
else:
    print(facts)

In [ ]:
print(f"{CYAN}{BOLD}Hook Trace Logs{RESET}\n")
logs = get_logs()
print(f"Total events logged: {len(logs)}\n")

if logs:
    tool_calls = [l for l in logs if l['event'] == 'tool_call']
    print(f"Tool calls made: {len(tool_calls)}\n")

    tool_counts = {}
    for log in tool_calls:
        tool = log['tool']
        tool_counts[tool] = tool_counts.get(tool, 0) + 1

    print(f"{USER_COLOR}Tools used:{RESET}")
    for tool, count in sorted(tool_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {tool}: {count} times")

    print(f"\n{USER_COLOR}Last 10 events:{RESET}")
    df = pd.DataFrame(logs[-10:])
    if 'args' in df.columns:
        df = df.drop(columns=['args'])
    print(df[['timestamp', 'event', 'tool']].to_string())
else:
    print(f"{INFO_COLOR}No logs yet. Run agent cells first.{RESET}")

In [ ]:
print(f"{CYAN}{BOLD}Structured Output: PlayerMatchStats Schema{RESET}\n")

from openai import OpenAI
import json
import re
from pydantic import ValidationError

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

historical_data = search_historical_stats(
    "Jamie Vardy Leicester 2015-16",
    n_results=25,
)

print(f"{INFO_COLOR}Retrieved historical data from RAG.{RESET}\n")

prompt = f"""
You are extracting structured football information.

IMPORTANT:
Only use information that appears in the supplied text.

If a value is unknown, use 0 for integers and "Unknown" for text.

Return ONLY valid JSON.

Historical Data:

{historical_data[:2500]}

Return EXACTLY:

{{
  "player_name": "Jamie Vardy",
  "team": "Leicester City",
  "opponent": "Various",
  "match_date": "2015-16 Season",
  "season": "2015/2016",
  "goals": 0,
  "assists": 0,
  "shots": 0,
  "shots_on_target": 0,
  "passes_completed": 0,
  "minutes_played": 0
}}
"""

try:

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b:free",
        messages=[
            {
                "role": "system",
                "content": "Return only JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
    )

    raw = response.choices[0].message.content.strip()

except Exception as e:

    print(f"{ERROR_COLOR}LLM Error:{RESET}", e)
    raw = ""

if raw:

    raw = raw.replace("```json", "")
    raw = raw.replace("```", "")
    raw = raw.strip()

    match = re.search(r"\{.*\}", raw, re.DOTALL)

    if match:
        raw = match.group()

    try:

        data = json.loads(raw)

        validated = PlayerMatchStats(**data)

        print(f"{SUCCESS_COLOR}✓ Schema Validation Passed{RESET}\n")

        print(f"Player: {validated.player_name}")
        print(f"Team: {validated.team}")
        print(f"Opponent: {validated.opponent}")
        print(f"Season: {validated.season}")
        print(f"Goals: {validated.goals}")
        print(f"Assists: {validated.assists}")
        print(f"Shots: {validated.shots}")
        print(f"Shots on Target: {validated.shots_on_target}")
        print(f"Passes Completed: {validated.passes_completed}")
        print(f"Minutes Played: {validated.minutes_played}")

    except ValidationError as e:

        print(f"{ERROR_COLOR}Schema Validation Failed{RESET}\n")
        print(e)

    except json.JSONDecodeError:

        print(f"{ERROR_COLOR}JSON Parsing Failed{RESET}\n")
        print(raw)

else:

    print(f"{ERROR_COLOR}Model returned no response.{RESET}")

In [ ]:
print(f"{CYAN}{BOLD}Structured Output: CurrentStandings Schema{RESET}\n")

standings_raw = get_standings()

if "error" in standings_raw.lower():
    print(f"{ERROR_COLOR}{standings_raw}{RESET}")

else:
    try:
        lines = standings_raw.splitlines()

        season = "2025-26"
        matchday = 38

        standings = []

        for line in lines:

            # Example:
            # 1. Liverpool | P:38 W:26 D:8 L:4 GF:86 GA:41 GD:+45 Pts:86

            if not line.strip():
                continue

            if not line[0].isdigit():
                continue

            try:
                left, right = line.split("|", 1)

                position = int(left.split(".")[0])
                team = left.split(".", 1)[1].strip()

                stats = {}

                for token in right.split():
                    if ":" in token:
                        k, v = token.split(":")
                        stats[k] = v.replace("+", "")

                standings.append(
                    TeamStanding(
                        position=position,
                        team=team,
                        played=int(stats.get("P", 0)),
                        won=int(stats.get("W", 0)),
                        drawn=int(stats.get("D", 0)),
                        lost=int(stats.get("L", 0)),
                        goals_for=int(stats.get("GF", 0)),
                        goals_against=int(stats.get("GA", 0)),
                        goal_difference=int(stats.get("GD", 0)),
                        points=int(stats.get("Pts", 0)),
                    )
                )

            except Exception:
                continue

        validated = CurrentStandings(
            season=season,
            matchday=matchday,
            top_team=standings[0].team if standings else "Unknown",
            top_scorer="See Cell 10 / Live Tool",
            standings=standings[:5],
            teams_in_top_4=[t.team for t in standings[:4]],
            teams_in_relegation=[t.team for t in standings[-3:]],
        )

        print(f"{SUCCESS_COLOR}Schema validation PASSED{RESET}\n")
        print(f"Season: {validated.season}")
        print(f"Top Team: {validated.top_team}")
        print(f"Top Scorer: {validated.top_scorer}")

        print("\nTop 4")
        for team in validated.teams_in_top_4:
            print("-", team)

        print("\nRelegation")
        for team in validated.teams_in_relegation:
            print("-", team)

        print("\nTop 5 Table\n")

        for team in validated.standings:
            print(
                f"{team.position}. {team.team} "
                f"{team.points} pts "
                f"(W{team.won} D{team.drawn} L{team.lost})"
            )

    except Exception as e:
        print(f"{ERROR_COLOR}{e}{RESET}")
        print("\nRaw standings:\n")
        print(standings_raw)

In [ ]:
print(f"{CYAN}{BOLD}Pydantic Validation - Intentional Failure Demo{RESET}\n")
print(f"{INFO_COLOR}Showing what happens when schema validation fails{RESET}\n")

bad_data_examples = [
    {
        'description': 'Missing required field (no goals)',
        'data': {
            'player_name': 'Test Player',
            'team': 'Test FC',
            'opponent': 'Other FC',
            'match_date': '2015-08-08',
            'season': '2015/2016',
            'assists': 5,
            'shots': 10,
            'shots_on_target': 5,
            'passes_completed': 30,
            'minutes_played': 90,
        }
    },
    {
        'description': 'Wrong type (goals as string)',
        'data': {
            'player_name': 'Test Player',
            'team': 'Test FC',
            'opponent': 'Other FC',
            'match_date': '2015-08-08',
            'season': '2015/2016',
            'goals': 'many',
            'assists': 5,
            'shots': 10,
            'shots_on_target': 5,
            'passes_completed': 30,
            'minutes_played': 90,
        }
    },
    {
        'description': 'Invalid value (negative goals)',
        'data': {
            'player_name': 'Test Player',
            'team': 'Test FC',
            'opponent': 'Other FC',
            'match_date': '2015-08-08',
            'season': '2015/2016',
            'goals': -5,
            'assists': 5,
            'shots': 10,
            'shots_on_target': 5,
            'passes_completed': 30,
            'minutes_played': 90,
        }
    },
]

for example in bad_data_examples:
    print(f"{USER_COLOR}Test: {example['description']}{RESET}")
    try:
        validated = PlayerMatchStats(**example['data'])
        print(f"{ERROR_COLOR}UNEXPECTED PASS - validation should have failed{RESET}")
    except ValidationError as e:
        print(f"{SUCCESS_COLOR}Correctly caught validation error:{RESET}")
        for error in e.errors():
            print(f"  Field: {error['loc']} | Error: {error['msg']}")
    print()

## Week 6 Summary

### Core Features Built
- Full agent loop — LLM decides which of 3 tools to use per question
- Multi-tool reasoning — agent chains tools for complex questions
- Session memory — follow-up questions reference earlier answers
- Hook logging — every tool call traced with timestamp
- Two Pydantic schemas validated: PlayerMatchStats and CurrentStandings
- OpenRouter primary with Ollama fallback

### Agent Routing Logic
| Question Type | Tool Used | Data Source |
|--------------|-----------|-------------|
| Current standings/scorers | get_standings, get_top_scorers | football-data.org |
| Historical player stats | search_historical_stats | StatsBomb ChromaDB |
| News and transfers | brave_search | Brave Search API |
| Multi-hop | Multiple tools | All three sources |

### Validation Results
- PlayerMatchStats schema: validated goals, assists, shots, minutes
- CurrentStandings schema: validated top 4, relegation, full table
- Intentional failure demo: showed 3 types of validation errors caught

### Next Week
Week 7 adds full 8-season ingestion, multi-model routing comparison, MCP server, and evaluation metrics.